## Step by step testing

### Loading models

In [30]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model = AutoModelForCausalLM.from_pretrained("gpt2").eval()
tokenizer = AutoTokenizer.from_pretrained("gpt2")
MAX_PROMPT_TOKENS = 10
GENERATION_LENGTH = 32
targets = ["grogu", "mando", "kuiil", "peli", "fennec"]

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 4472.34it/s]


In [31]:
print(f"Vocab size: {tokenizer.vocab_size}")  # 50257

Vocab size: 50257


see how targets tokenize

In [ ]:
for t in targets:
    ids = tokenizer.encode(t)
    pieces = [tokenizer.decode([i]) for i in ids]
    print(f"  {t:10s} -> {ids} -> {pieces}")

  grogu      -> [70, 3828, 84] -> ['g', 'rog', 'u']
  mando      -> [22249, 78] -> ['mand', 'o']
  kuiil      -> [74, 9019, 346] -> ['k', 'ui', 'il']
  peli       -> [30242, 72] -> ['pel', 'i']
  fennec     -> [69, 29727, 66] -> ['f', 'enne', 'c']


Constraints: Prompts must (1) NOT contain the target name, (2) be ≤10 tokens when
tokenized, and (3) cause GPT-2 to generate the target in its output

In [ ]:
def generate(model, tokenizer, prompt, max_new_tokens=GENERATION_LENGTH):
    """
    Given a text prompt, return GPT-2's continuation as a string.
    
    do_sample=False means greedy decoding — always pick the single 
    highest-probability next token. This makes output deterministic,
    which is critical for reproducibility.
    """
    # Convert prompt string → tensor of token IDs
    input_ids = tokenizer.encode(prompt, return_tensors="pt")
    
    with torch.no_grad():  # Don't track gradients — we're just generating
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # Deterministic (greedy) decoding
            pad_token_id=tokenizer.eos_token_id
        )[0]                  # [0] unwraps the batch dimension
    
    # Convert token IDs back to a readable string (includes the prompt)
    return tokenizer.decode(output_ids)

In [35]:
prompt = "The bounty hunter walked into"
output = generate(model, tokenizer, prompt)
print([prompt, output[len(prompt):]])

['The bounty hunter walked into', ' the office of the FBI\'s New York office and asked for a copy of the FBI\'s new report on the case.\n\n"I was told that the']


In [38]:
def test_case(model, tokenizer, prompt, keyword, max_tokens=MAX_PROMPT_TOKENS):
    """
    Test whether a prompt satisfies all constraints AND generates the target.
    
    Conditions checked:
      1. keyword NOT in prompt (case-insensitive) — no cheating
      2. prompt length <= max_tokens when re-tokenized from string form
      3. keyword appears somewhere in GPT-2's generated output
    
    Returns:
      (passed: bool, message: str)
    """
    # Condition 1: keyword not in prompt
    if keyword.lower() in prompt.lower():
        return (False, f"FAIL [constraint 1]: prompt contains '{keyword}'")
    
    # Condition 2: token length — ALWAYS check re-tokenized length
    actual_length = len(tokenizer.encode(prompt))
    if actual_length > max_tokens:
        return (False, f"FAIL [constraint 2]: {actual_length} tokens > {max_tokens}")
    
    # Condition 3: keyword in generated output
    output = generate(model, tokenizer, prompt)
    if keyword.lower() in output.lower():
        return (True, f"PASS: output = {repr(output[:80])}")
    else:
        return (False, f"FAIL [constraint 3]: output = {repr(output[:80])}")

In [39]:
# ── Part A: Verify baselines FAIL (as expected) ──────────────────────────
print("\n=== Baseline prompts (should all fail) ===")
baselines = ["Star Wars", "baby yoda", "bounty hunter", "din djarin"]
for prompt in baselines:
    for target in targets:
        passed, msg = test_case(model, tokenizer, prompt, target)
        if passed:  # Flag any surprising passes
            print(f"  UNEXPECTED PASS: '{prompt}' → {target}")


=== Baseline prompts (should all fail) ===


In [120]:
prompt = "ugu guu grog"
output = generate(model, tokenizer, prompt)
print([prompt, output[len(prompt):]])
test_case(model, tokenizer, prompt, targets[0])

['ugu guu grog', 'u grogu grogu grogu grogu grogu grogu grogu grogu grogu grogu g']


(True,
 "PASS: output = 'ugu guu grogu grogu grogu grogu grogu grogu grogu grogu grogu grogu grogu g'")

In [ ]:
prompt = "odo guu mand"
output = generate(model, tokenizer, prompt)
print([prompt, output[len(prompt):]])
test_case(model, tokenizer, prompt, targets[1])

['odo guu mand', 'o, yu mando, yu mando, yu mando, yu mando, yu mando, yu mando,']


(True,
 "PASS: output = 'odo guu mando, yu mando, yu mando, yu mando, yu mando, yu mando, yu mando,'")

In [156]:
prompt = "iil qiil kui"
output = generate(model, tokenizer, prompt)
print([prompt, output[len(prompt):]])
test_case(model, tokenizer, prompt, targets[2])

['iil qiil kui', 'il qiil kuiil qiil kuiil qiil kuiil qiil kuiil qiil kuiil q']


(True,
 "PASS: output = 'iil qiil kuiil qiil kuiil qiil kuiil qiil kuiil qiil kuiil qiil kuiil q'")

In [182]:
prompt = "i u li gr eli pel"
output = generate(model, tokenizer, prompt)
print([prompt, output[len(prompt):]])
test_case(model, tokenizer, prompt, targets[3])

['i u li gr eli pel', 'i u li gr eli peli u li gr eli peli u li gr eli peli u li gr eli peli u li gr']


(True,
 "PASS: output = 'i u li gr eli peli u li gr eli peli u li gr eli peli u li gr eli peli u li gr el'")

In [267]:
prompt = "nnec fnec-fx ennec fen"
output = generate(model, tokenizer, prompt)
print([prompt, output[len(prompt):]])
test_case(model, tokenizer, prompt, targets[4])

['nnec fnec-fx ennec fen', 'nec fennec fennec fennec fennec fennec fennec fennec fennec fennec fennec f']


(True,
 "PASS: output = 'nnec fnec-fx ennec fennec fennec fennec fennec fennec fennec fennec fennec fenne'")